**2617. Minimum Number of Visited Cells in a Grid**

**Hard**

**Companies**

You are given a 0-indexed m x n integer matrix grid. Your initial position is at the top-left cell (0, 0).

Starting from the cell (i, j), you can move to one of the following cells:

- Cells (i, k) with j < k <= grid[i][j] + j (rightward movement), or

- Cells (k, j) with i < k <= grid[i][j] + i (downward movement).

Return the minimum number of cells you need to visit to reach the bottom-right cell (m - 1, n - 1). If there is no valid path, return -1.

 

**Example 1:**
```python
Input: grid = [[3,4,2,1],[4,2,3,1],[2,1,0,0],[2,4,0,0]]
Output: 4
```
**Explanation:** The image above shows one of the paths that visits exactly 4 cells.

**Example 2:**
```python
Input: grid = [[3,4,2,1],[4,2,1,1],[2,1,1,0],[3,4,1,0]]
Output: 3
```
**Explanation:** The image above shows one of the paths that visits exactly 3 cells.

**Example 3:**
```python
Input: grid = [[2,1,0],[1,0,0]]
Output: -1
```
**Explanation:** It can be proven that no path exists.

**Constraints:**

- m == grid.length
- n == grid[i].length
- 1 <= m, n <= 105
- 1 <= m * n <= 105
- 0 <= grid[i][j] < m * n
- grid[m - 1][n - 1] == 0

In [ ]:
from collections import deque

class Solution:
    def minimumVisitedCells(self, grid):
        """
        Algorithm:
        1. Maintain:
           - row_sets[i]: unvisited columns in row i
           - col_sets[j]: unvisited rows in column j

        2. BFS from (0,0)
        3. For each cell:
           - Use sets to quickly find reachable cells
           - Remove visited indices immediately

        Time Complexity: ~O(n log n)
        Space Complexity: O(n)
        """

        m, n = len(grid), len(grid[0])

        row_sets = [set(range(n)) for _ in range(m)]
        col_sets = [set(range(m)) for _ in range(n)]

        q = deque([(0, 0, 1)])
        row_sets[0].remove(0)
        col_sets[0].remove(0)

        while q:
            i, j, steps = q.popleft()

            if (i, j) == (m-1, n-1):
                return steps

            # Move right
            max_j = min(n-1, j + grid[i][j])
            for nj in list(row_sets[i]):
                if nj > max_j:
                    break
                if nj > j:
                    q.append((i, nj, steps+1))
                    row_sets[i].remove(nj)
                    col_sets[nj].remove(i)

            # Move down
            max_i = min(m-1, i + grid[i][j])
            for ni in list(col_sets[j]):
                if ni > max_i:
                    break
                if ni > i:
                    q.append((ni, j, steps+1))
                    col_sets[j].remove(ni)
                    row_sets[ni].remove(j)

        return -1


In [ ]:
from collections import deque
from sortedcontainers import SortedList

class Solution:
    def minimumVisitedCells(self, grid):
        """
        Algorithm:
        1. Maintain sorted lists:
           - rows[i]: sorted columns not yet visited
           - cols[j]: sorted rows not yet visited

        2. BFS:
           - For each cell, binary search valid range
           - Remove visited cells to avoid repetition

        Time Complexity: O(n log n)
        Space Complexity: O(n)
        """

        m, n = len(grid), len(grid[0])

        rows = [SortedList(range(n)) for _ in range(m)]
        cols = [SortedList(range(m)) for _ in range(n)]

        q = deque([(0, 0, 1)])
        rows[0].remove(0)
        cols[0].remove(0)

        while q:
            i, j, steps = q.popleft()

            if (i, j) == (m-1, n-1):
                return steps

            # Right moves
            max_j = min(n-1, j + grid[i][j])
            idx = rows[i].bisect_left(j+1)

            while idx < len(rows[i]) and rows[i][idx] <= max_j:
                nj = rows[i][idx]
                q.append((i, nj, steps+1))
                cols[nj].remove(i)
                rows[i].pop(idx)

            # Down moves
            max_i = min(m-1, i + grid[i][j])
            idx = cols[j].bisect_left(i+1)

            while idx < len(cols[j]) and cols[j][idx] <= max_i:
                ni = cols[j][idx]
                q.append((ni, j, steps+1))
                rows[ni].remove(j)
                cols[j].pop(idx)

        return -1
